In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Decoding and Generation

The language models of this chapter have been judged by how well they
predict the next token. To write text, a model must also choose one token
at each step from the distribution it returns, and this choice matters more
than one might expect: different rules make the same model write very
different text. The simplest rule, tried in that section, fell
short: always taking the most probable token tends to run into loops. How,
then, should the choice be made?

The rule that turns next-token distributions into sequences is called
*decoding* [@Graves.2013; @Holtzman.Buys.Du.ea.2020], and decoding rules
come in two families. *Maximization* searches for a sequence of high
probability under the model. *Sampling* draws each token from the model's
distribution, after cutting off the part of it that cannot be trusted. We
develop both in turn. All of these rules see the model only through its
next-token distribution, so they apply unchanged to any language model,
recurrent or not.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The Decoding Problem

A trained language model supplies conditionals: writing $x_{<t}$ for the
prefix $x_1, \ldots, x_{t-1}$, it turns any prefix into a distribution
$P(x_t \mid x_{<t})$ over the next token. By the chain rule of probability
the equation, these conditionals assign every sequence the
probability $P(x_1, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_{<t})$.

Decoding asks the inverse question: given the conditionals, which sequence
should we output? The obvious answer is the *most probable* one.
Unfortunately, finding it is out of the question. With vocabulary
$\mathcal{V}$ and $T$ tokens to generate there are $|\mathcal{V}|^T$
candidates. For our 1,027 tokens and a 50-token continuation this is
$1027^{50} > 10^{150}$, far too many to enumerate, let alone compare. A
practical decoder therefore extends only a few prefixes at each step and
must decide which ones to keep.

Even if we could find it, the most probable sequence is often not what we
want. When the input admits only a narrow set of acceptable outputs, as in
transcribing speech, a high-probability output is a good target. When many
continuations are acceptable, as in writing a story, each of them is
individually improbable, and a model that has learned this spreads its
probability across them. Insisting on the single most probable output then
throws away the variety the model has learned. Sampling keeps it, because
it draws the output from the model's distribution instead of maximizing it.

the figure sorts the decoding rules into these two
families. Maximization methods, greedy decoding and beam search, are common
when the acceptable outputs are few. Sampling methods, which first reshape
or truncate the distribution, are the default for open-ended generation
such as dialogue and stories.

![Two families of decoding rules. Searching for high-probability sequences suits tasks with few acceptable outputs; sampling preserves diversity in open-ended generation.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-rnn-decoding-taskmap.svg)

### A Model to Decode From

We decode from the concise RNN language model of that section:
an embedding of dimension 512, a recurrent layer with 128 hidden units,
dropout on the embeddings and the hidden states, and an output layer over
the 1,027 BPE tokens of *The Time Machine*, trained for ten epochs on
windows of 32 tokens with gradient clipping. We retrain it with the same
settings and print its validation perplexity, which should be close to the
value reached there.

In [ ]:
data = d2l.TimeMachine(batch_size=1024, num_steps=32)

In [ ]:
rnn = d2l.RNN(num_inputs=512, num_hiddens=128)
model = d2l.RNNLM(rnn, vocab_size=len(data.vocab), lr=4, dropout=0.5,
                  emb_dropout=0.1)

In [ ]:
trainer = d2l.Trainer(max_epochs=10, gradient_clip_val=1, num_gpus=1)
trainer.fit(model, data)
print(f'validation perplexity {math.exp(trainer.history["val_loss"][-1]):.1f}')

Every decoding rule in this section sees the model through one interface:
given the token ids so far, return the logits of the next token. We wrap
the trained network in a function `step_fn` of this form. It returns a
NumPy vector, so that the decoding rules can be written once, in NumPy, for
all frameworks. The interface carries no hidden state, and this is
deliberate: any model that predicts next tokens can fill it, including one
that has no fixed-size state to carry.

In [ ]:
@nnx.jit
def forward(model, X):
    return model(X)

model_eval = nnx.view(model, deterministic=True, raise_if_not_found=False)

def step_fn(ids):  # Token ids in, NumPy logits for the next token out
    padded = ids + [0] * (-len(ids) % 32)  # Few distinct lengths to compile
    logits = forward(model_eval, d2l.tensor([padded]))  # Dropout off
    return d2l.numpy(logits)[0, len(ids) - 1]

`nnx.jit` compiles the forward pass once per input shape, so we pad each
prefix with zeros to a multiple of 32 tokens and compile only a few shapes.
The network reads the sequence from left to right, so the padding does not
change the logits at the last real position, which is the one we read.

In return, each call runs the network over the *entire* prefix to produce
one vector of logits. For a recurrent network this is wasteful, and we
count the waste at the end of the section.

## Greedy Decoding

The simplest decoder commits, at every step, to the most probable token:

$$x_t = \operatorname*{argmax}_{x \in \mathcal{V}} P(x \mid x_{<t}).$$

Score, take the argmax, append, repeat. Greedy decoding costs one model
evaluation per token and is deterministic: the same prefix always yields
the same continuation. To print continuations, `show` decodes token ids
and quotes the text, with line breaks written as `\n` and unprintable
characters replaced by U+FFFD.

In [ ]:
def show(ids):  # Decoded text as a literal; control characters -> U+FFFD
    text = data.tokenizer.decode(ids)
    return repr(''.join(c if c.isprintable() or c == '\n' else '\ufffd'
                        for c in text))

def decode_greedy(text, num_tokens=50):
    ids = data.tokenizer.encode(text)
    for _ in range(num_tokens):
        ids.append(int(step_fn(ids).argmax()))
    return ids

for text in ('The Time Traveller', 'It seemed to me that'):
    d2l.print_wrapped(show(decode_greedy(text)))

Both continuations end in a loop, as the greedy sample of
that section did. Two flaws are built into the rule. The first
is *myopia*: a sequence
of locally best tokens need not be the best sequence. Suppose the model
gives the first token the probabilities $P(A) = 0.5$ and $P(B) = 0.4$.
After $A$ the model is unsure, and its most probable second token has
probability $0.3$; after $B$ it is confident, with $0.9$ for its favorite.
Greedy decoding picks $A$ and ends with a sequence of probability $0.5
\times 0.3 = 0.15$, whereas starting with $B$ gives $0.4 \times 0.9 = 0.36$.
Greedy decoding cannot see that the more probable first token leads into an
uncertain second step.

The second flaw is the loop itself. Greedy decoding feeds its own argmax
back in as context, and nothing in it is random: the next token is a
function of the hidden state, and the next state is a function of the state
and that token. Once the state has settled into a cycle, no chance event
can dislodge it. Is this a defect of our small model? Unfortunately not.
@Holtzman.Buys.Du.ea.2020 documented this *degeneration* in
language models about a thousand times larger than ours:
maximization-based decoding of open-ended text produces output that is
repetitive and measurably unlike human text, however good the model.

## Beam Search

Greedy decoding commits immediately; exhaustive search never commits at all.
*Beam search* lies between them, with a single setting, the *beam width*
$w$: it keeps the $w$ most probable prefixes alive at every step. At the
first step it selects the $w$ most probable tokens. After that, each of the
$w$ surviving prefixes is extended by every token in the vocabulary, the
resulting $w \cdot |\mathcal{V}|$ candidates are scored by their cumulative
log-probability, and the best $w$ survive. Greedy decoding is the case
$w = 1$. Exhaustive search is the limit in which $w$ exceeds the number of
candidates and nothing is ever pruned.

![Beam search with beam width 2 over a five-token vocabulary. At every step each surviving prefix is expanded over the whole vocabulary and only the two highest-scoring prefixes are kept; the survivors here are $A$ and $C$, then $AB$ and $CE$, then $ABD$ and $CED$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-rnn-beam-tree.svg)

the figure traces an example with $w = 2$ over the
vocabulary $\mathcal{V} = \{A, B, C, D, E\}$ and three steps. Every
probability is conditional on the given prefix, which we leave out of the
notation. Suppose that the two most probable first tokens are $A$ and $C$;
they form the initial beam. At the second step we compute, for every
$x' \in \mathcal{V}$, the scores of the ten extensions

$$
\begin{aligned}
P(A, x') &= P(A)\, P(x' \mid A),\\
P(C, x') &= P(C)\, P(x' \mid C),
\end{aligned}
$$

and keep the two largest, say for $AB$ and $CE$. The third step repeats the
calculation from these survivors and keeps, say, $ABD$ and $CED$. Instead
of the $5^3 = 125$ sequences of exhaustive search, beam search scored
$5 + 10 + 10 = 25$. In general it costs $O(w |\mathcal{V}| T)$
for $T$ steps, $w$ times as much as greedy decoding and exponentially less
than exhaustive search.

One correction is needed before finished candidates of different lengths
can be compared. The log-probability of a token is never positive, so the
score of a sequence can only fall as it grows, and comparing raw scores
favors short candidates. This matters once an end-of-sequence token lets
some candidates stop early. The standard remedy is *length normalization*
[@Wu.Schuster.Chen.ea.2016]: rank a candidate that adds $T$ tokens to
a prefix of $t_0$ tokens by

$$\frac{1}{T^\beta} \sum_{t=t_0+1}^{t_0+T} \log P(x_t \mid x_{<t}),$$

where the exponent $\beta$, typically around 0.75, interpolates between no
correction ($\beta = 0$) and the average log-probability per token
($\beta = 1$). In the code below it is the argument `alpha`, the name used
in @Wu.Schuster.Chen.ea.2016 .

The implementation below follows this procedure. Finished candidates, those
that produced `eos_id`, pass from round to round unchanged; live ones are
expanded and rescored. Since the best $w$ candidates contain at most $w$
continuations of any single prefix, it suffices to expand each prefix by
its $w$ most probable tokens rather than by all $|\mathcal{V}|$ of them.

In [ ]:
def beam_search(step_fn, prefix, num_tokens, beam_size=4,
                alpha=0.75, eos_id=None):
    """Length-normalized beam search (score = log P / len^alpha).
    Returns candidates sorted best-first; [0][1] is the argmax sequence."""
    def log_softmax(logits):
        logits = [float(l) for l in logits]
        m = max(logits)
        lse = m + math.log(sum(math.exp(l - m) for l in logits))
        return [l - lse for l in logits]
    score = lambda logp, ids: logp / max(1, len(ids) - len(prefix))**alpha
    beams = [(0.0, list(prefix), False)]  # (log-prob, sequence, finished)
    for _ in range(num_tokens):
        if all(done for _, _, done in beams):
            break
        candidates = [b for b in beams if b[2]]  # Finished pass through
        for logp, ids, _ in (b for b in beams if not b[2]):
            logprobs = log_softmax(step_fn(ids))
            best = sorted(range(len(logprobs)),
                          key=lambda i: -logprobs[i])[:beam_size]
            candidates += [(logp + logprobs[i], ids + [i], i == eos_id)
                           for i in best]
        beams = sorted(candidates, key=lambda b: score(b[0], b[1]),
                       reverse=True)[:beam_size]
    return sorted([(score(logp, ids), ids) for logp, ids, _ in beams],
                  reverse=True)

We decode the two prefixes from before with $w = 1$, which reproduces
greedy decoding, and with $w = 4$, and print the score
the equation of the winning candidate. Our text has no
end-of-sequence token, so every candidate has 30 tokens and the length
correction changes no ranking here: the score is the log-probability
divided by $30^{0.75} \approx 12.8$.

In [ ]:
for text in ('The Time Traveller', 'It seemed to me that'):
    prefix = data.tokenizer.encode(text)
    for w in (1, 4):
        score, ids = beam_search(step_fn, prefix, 30, beam_size=w)[0]
        d2l.print_wrapped(f'w={w}, score {score:.2f}: {show(ids)}')

The wider beam usually finds continuations that the model scores higher
than the greedy ones, but it cannot guarantee it: beam search is still an
approximate search, and it can prune the prefix that greedy decoding would
have completed. Whether the higher-scoring text is also *better* text is a
separate question. In large language models, beam
search on open-ended text produces output far more repetitive than human
text [@Holtzman.Buys.Du.ea.2020], and we measure below how our
maximizers compare with sampling. Where the acceptable outputs are few, as
in speech recognition and machine translation, beam search with a small
width, typically between 4 and 10, is the standard decoder
[@Sutskever.Vinyals.Le.2014; @Wu.Schuster.Chen.ea.2016].

## Sampling

If we trust the model's distribution, the natural way to generate is to
sample from it: draw $x_t \sim P(x_t \mid x_{<t})$, append, repeat. Text
produced this way is distributed as the model believes text should be, with
the variety that maximization throws away. The problem lies in the tail.
Our model spreads small probabilities over a thousand tokens, a large model
over a hundred thousand. Each unlikely token is negligible on its own, but
their combined probability at each step is not. Sampling therefore
regularly picks a token that the model itself considers nearly impossible,
and one absurd token, fed back as context, can derail everything after it.
This is the compounding of errors in a rollout (that section),
with tokens in place of numbers. @Holtzman.Buys.Du.ea.2020 call the
culprit the *unreliable tail*: the model's estimates are trustworthy for
its most probable tokens and mostly noise far down the ranking. The fix is
to reshape or truncate the distribution before sampling, and each of the
settings below does this in its own way.

**Temperature** divides every
logit $o_x$ by $\tau > 0$ before the softmax:

$$P_\tau(x \mid x_{<t}) = \frac{\exp(o_x / \tau)}{\sum_{x' \in \mathcal{V}} \exp(o_{x'} / \tau)} \propto P(x \mid x_{<t})^{1/\tau}.$$

Values of $\tau$ below 1 sharpen the distribution, down to greedy decoding
as $\tau \to 0$, and values above 1 flatten it toward uniform. Temperature
is a blunt tool. It reshapes the head and the tail together, so cooling the
distribution enough to suppress the tail also removes legitimate variety at
the head.

**Top-$k$** truncates by count: keep the $k$ most probable tokens,
renormalize, and sample
[@Fan.Lewis.Dauphin.2018; @Radford.Wu.Child.ea.2019]. **Top-$p$** (or
*nucleus*) sampling truncates by probability mass: keep the smallest set of
most probable tokens whose total probability reaches $p$, for example
$p = 0.9$ [@Holtzman.Buys.Du.ea.2020]. The difference matters because
the model's confidence changes from step to step. Where the model is nearly
certain of the next token, a fixed $k = 20$ admits nineteen poor options;
where dozens of words are plausible, the same $k$ may cut off real variety.
Top-$p$ adapts: it keeps few tokens when the head of the distribution is
heavy and many when it is flat. It has its own failure on peaked
distributions: if the top token holds 0.95 of the mass, top-$p$ with
$p = 0.9$ keeps one token and acts greedily even where the runner-up was a
good choice.

**Min-$p$** sets the cutoff relative to the model's confidence
[@Nguyen.Baker.Neo.ea.2025]: keep every token whose probability is at
least $p_{\min}$ times that of the most probable token, for example
$p_{\min} = 0.05$. When the model is sure, the bar is high and few tokens
pass; when it is unsure, the bar drops and more of them survive.
@Nguyen.Baker.Neo.ea.2025 find that this relative rule degrades
less than top-$p$ at high temperatures, where an absolute rule admits much
of the flattened tail.

### A Unified Sampler

All four settings act on the same object, the next-token distribution, so
one function can implement them. `sample_next` turns a vector of logits
into a token id. At temperature 0 it returns the argmax, which is greedy
decoding. Otherwise it computes $P_\tau$ of the equation, sorts
the tokens by probability, and applies the optional top-$k$, top-$p$, and
min-$p$ cutoffs. Each cutoff only decides how many tokens of the sorted
order to keep, and the function keeps the smallest of these numbers. It
then renormalizes the kept probabilities and draws one token. It takes a
NumPy random generator `rng` and creates a fresh one if none is given.

In [ ]:
def sample_next(logits, temperature=1.0, k=None, p=None, min_p=None,
                rng=None):
    """Draw a token id from 1-D NumPy logits; temperature 0 is greedy.
    Top-k, top-p, and min-p truncate the temperature-scaled distribution."""
    logits = np.asarray(logits, dtype=np.float64)
    if temperature == 0:
        return int(logits.argmax())
    probs = np.exp((logits - logits.max()) / temperature)
    order = np.argsort(-probs)               # token ids, most probable first
    probs = probs[order] / probs.sum()
    keep = len(probs)
    if k is not None:                        # top-k: a fixed count
        keep = min(keep, k)
    if p is not None:                        # top-p: smallest head of mass p
        keep = min(keep, int(np.searchsorted(probs.cumsum(), p)) + 1)
    if min_p is not None:                    # min-p: a fraction of the top
        keep = min(keep, int((probs >= min_p * probs[0]).sum()))
    rng = np.random.default_rng() if rng is None else rng
    return int(order[rng.choice(keep, p=probs[:keep] / probs[:keep].sum())])

`generate` is the loop we have been writing by hand: query `step_fn`,
choose with `sample_next` (extra keyword arguments pass through to it),
append, and stop early if the end-of-sequence token `eos_id` appears.
Open-ended generation has no such token, so our examples run for a fixed
number of tokens.

In [ ]:
def generate(step_fn, prefix, num_tokens, eos_id=None, **strategy):
    """Extend prefix by up to num_tokens ids, each chosen by sample_next
    from step_fn(ids), the next-token logits; stop early at eos_id."""
    ids = list(prefix)
    for _ in range(num_tokens):
        ids.append(sample_next(step_fn(ids), **strategy))
        if eos_id is not None and ids[-1] == eos_id:
            break
    return ids

A toy distribution makes the cutoffs concrete. Five tokens have
probabilities 0.45, 0.25, 0.15, 0.10, and 0.05. Top-$k$ with $k = 2$ can
only emit tokens 0 and 1. Top-$p$ with $p = 0.8$ keeps three tokens,
since the cumulative probability first reaches 0.8 at the third (0.85).
Min-$p$ with $p_{\min} = 0.5$ keeps the tokens with probability at least $0.5
\times 0.45 = 0.225$, again tokens 0 and 1. We draw 1,000 tokens under
each rule and list the tokens that occur.

In [ ]:
logits = np.log([0.45, 0.25, 0.15, 0.10, 0.05])
print('greedy:', sample_next(logits, temperature=0))
for cutoff in (dict(k=2), dict(p=0.8), dict(min_p=0.5)):
    drawn = {sample_next(logits, **cutoff) for _ in range(1000)}
    print(cutoff, 'tokens drawn:', sorted(drawn))

Each rule draws exactly the tokens it keeps: tokens 0 and 1 under top-$k$
and min-$p$, tokens 0 to 2 under top-$p$.

### Cutoffs on a Trained Model

What do the cutoffs do to a real distribution? We compute our model's
next-token distribution after "The Time Traveller", sort it by decreasing
probability, and plot it on logarithmic axes. On this plot each rule is a
single line.
Top-$k$ is a vertical line at rank $k$, wherever the probabilities lie.
Top-$p$ is a vertical line at the rank where the cumulative probability
reaches $p$, so its position depends on the shape of the head. Min-$p$ is a
*horizontal* line at $p_{\min}$ times the top probability: every token
above it is kept, and the line moves up and down with the model's
confidence. Everything to the right of the vertical lines, or below the
horizontal one, is the tail that truncation removes.

In [ ]:
def next_probs(text):  # Next-token probabilities after text, sorted
    logits = step_fn(data.tokenizer.encode(text))
    probs = np.exp(logits - logits.max())
    return np.sort(probs / probs.sum())[::-1]

def cutoffs(probs, p=0.9, min_p=0.05):  # Tokens kept by top-p and min-p
    return (int(np.searchsorted(probs.cumsum(), p)) + 1,
            int((probs >= min_p * probs[0]).sum()))

probs = next_probs('The Time Traveller')
n_p, n_m = cutoffs(probs)
lo, hi = probs[-1], probs[0]
d2l.plot([np.arange(1, len(probs) + 1), [20, 20], [n_p, n_p],
          [1, len(probs)]], [probs, [lo, hi], [lo, hi], [0.05 * hi] * 2],
         xlabel='token rank', ylabel='probability', xscale='log',
         yscale='log', figsize=(5.5, 3),
         legend=['model', 'top-k, k=20', f'top-p=0.9: keep {n_p}',
                 f'min-p=0.05: keep {n_m}'])

On the logarithmic axes the curve declines steadily, from a top
probability of about 0.2 to about $10^{-3}$ near rank 100, and falls
faster still over the last few hundred ranks. Most of the vocabulary lies
in this tail of tokens with tiny probabilities, the tail that all three
rules cut. To compare the rules, we also apply them after the
shorter prefix "The Time", which the novel continues with " Traveller" in
11 of its 13 occurrences, so the model should be nearly sure of the next
token. We count the tokens each rule keeps after both prefixes.

In [ ]:
for text in ('The Time Traveller', 'The Time'):
    probs = next_probs(text)
    n_p, n_m = cutoffs(probs)
    print(f'{text!r}: top probability {probs[0]:.2f}; kept: top-k 20, '
          f'top-p {n_p}, min-p {n_m}')

Top-$k$ keeps 20 tokens whatever the distribution. Both other rules
tighten when the model is sure: after "The Time" the top probability is
high, both keep far fewer tokens, and min-$p$ keeps only a handful. They
differ when the model is unsure. After "The Time Traveller", top-$p$ must
reach far into the tail to collect 90% of the mass and keeps well over a
hundred tokens, most of them with probability below one percent. Min-$p$
looks only at the top probability: it keeps the tokens within a factor of
twenty of the best, a few dozen at most. Which of the two is right, the
counts cannot say.

### Strategies Compared on One Prefix

We now read the text that each setting produces, starting with temperature
alone at $\tau = 0.5$, $1$, and $2$.

In [ ]:
prefix = data.tokenizer.encode('The Time Traveller')
for tau in (0.5, 1.0, 2.0):
    out = generate(step_fn, prefix, 50, temperature=tau)
    d2l.print_wrapped(f'tau={tau}: {show(out)}')

At $\tau = 0.5$ the text stays close to the model's most probable tokens.
At $\tau = 1$ it is more varied and occasionally strays into invented
words. At $\tau = 2$ the flattened distribution draws so many tail tokens
that the text stops being English; the tokens may even combine into bytes
that do not form valid characters. Next we run the main strategies on the
same prefix, each with its typical setting.

In [ ]:
strategies = {'greedy': dict(temperature=0), 'tau=1': dict(),
              'top-k=20': dict(k=20), 'top-p=0.9': dict(p=0.9),
              'min-p=0.05': dict(min_p=0.05)}
for name, strategy in strategies.items():
    out = generate(step_fn, prefix, 50, **strategy)
    d2l.print_wrapped(f'{name:>10}: {show(out)}')

Our small model cannot make any of these continuations good, and one sample
per rule is thin evidence. Greedy decoding is deterministic: rerunning the
cell reproduces its text exactly, while every sampler gives new text each
time. Only greedy decoding loops. Among the samplers these lines show no
ranking, since invented words appear with and without truncation. The
truncated samplers are designed to keep the variety of sampling and cut the
tail on which pure sampling trips, and a temperature combined with top-$p$
or min-$p$ is a common default for open-ended generation.

## Evaluation and Cost

### The Quality of Generated Text

How can we decide, beyond reading samples, which strategy writes better
text? Perplexity is of little help. It evaluates the *model*: how well its
conditionals predict text that people wrote. It says nothing about a
particular generated sample, and scoring a sample by its probability under
the model would reward maximization by definition. A strategy is not better
because its output is more probable; sampling deviates from the most
probable output on purpose.

Automated statistics can still catch specific failures. A simple one for
repetition is *distinct-3*, the fraction of distinct 3-grams among all
3-grams of a continuation: it is 1 when no three consecutive tokens ever
recur and close to 0 for text stuck in a loop. We decode 100 tokens after 20
random six-token prefixes from the validation text with greedy decoding,
beam search with $w = 4$, and the samplers of the previous section, and
report two averages:
the log-probability per generated token under the model, and distinct-3.

In [ ]:
def log_prob(ids, start):  # Mean log-probability of ids[start:]
    lps = []
    for t in range(start, len(ids)):
        logits = step_fn(ids[:t])
        lse = logits.max() + np.log(np.exp(logits - logits.max()).sum())
        lps.append(logits[ids[t]] - lse)
    return np.mean(lps)

def distinct(ids, n=3):  # Fraction of distinct n-grams in ids
    grams = [tuple(ids[i:i + n]) for i in range(len(ids) - n + 1)]
    return len(set(grams)) / len(grams)

val = d2l.numpy(data.X[data.num_train:data.num_train + data.num_val])
rows = np.random.default_rng().choice(len(val), 20, replace=False)
prefixes = [[int(i) for i in val[r, :6]] for r in rows]
decoders = {
    'greedy': lambda pre: generate(step_fn, pre, 100, temperature=0),
    'beam, w=4': lambda pre: beam_search(step_fn, pre, 100, beam_size=4)[0][1],
    'tau=1': lambda pre: generate(step_fn, pre, 100),
    'top-k=20': lambda pre: generate(step_fn, pre, 100, k=20),
    'top-p=0.9': lambda pre: generate(step_fn, pre, 100, p=0.9),
    'min-p=0.05': lambda pre: generate(step_fn, pre, 100, min_p=0.05)}
for name, decode in decoders.items():
    outs = [decode(pre) for pre in prefixes]
    print(f'{name:>10}: log-prob per token '
          f'{np.mean([log_prob(o, 6) for o in outs]):.2f}, distinct-3 '
          f'{np.mean([distinct(o[6:]) for o in outs]):.2f}')

The two averages disagree. The two maximizers produce continuations with a
clearly higher log-probability per token than any of the samplers. Beam
search is usually ahead of greedy decoding, by a margin that varies from
run to run and occasionally reverses. Yet the samples are far more varied:
their distinct-3 is close to 1, while the maximizers repeat most of their
3-grams. Maximization raises the probability of the text and lowers its
variety, the degeneration described by @Holtzman.Buys.Du.ea.2020
on a small scale.

Among the samplers, variety does not separate the rules, but probability
does. Pure sampling at $\tau = 1$ has the lowest log-probability per token,
about $-4$. The three truncated samplers lie between it and the
maximizers, and their continuations are as varied as those of pure
sampling. Top-$p$ at $0.9$ stays nearest to pure sampling, which is what
the counts above suggested, since it keeps the most tokens; top-$k$ and
min-$p$ cut deeper, and their text is about a nat per token more probable. Cutting the tail does
what it was designed to do.

Such statistics detect particular failures; they do not say whether the
text is good. The final judge of open-ended generation is a human reader:
can people tell the model's text from text that people wrote, and which do
they prefer? Tasks with a reference answer, such as translation, are easier
to score automatically, by the overlap between the output and reference
translations. In short, perplexity selects the model, and human judgment,
or a proxy for it, selects the decoding strategy.

### The Cost of Generation

We built `step_fn` without a state so that it would fit any model, and so
it runs the network over the whole prefix for every token it generates. A
continuation of $T$ tokens thus costs $O(T^2)$ updates of the recurrent
cell. Generation is also *sequential*: during training, all positions of a
known sequence are processed together, but token $t + 1$ cannot be scored
before token $t$ has been chosen. The sequential part cannot be avoided.
The quadratic part, for a recurrent network, is pure waste. The hidden
state already summarizes the prefix, so carrying it forward, as `predict`
in that section does, makes each new token cost one cell
update, independent of the length of the prefix.

This constant cost and memory per generated token is what a fixed-size
state provides, and a reason for the interest in recurrent language models.
The same state is the model's limit: whatever it remembers of the prefix
must fit into 128 numbers (that section). No
decoding rule can put into the text what the model's distribution does not
contain. The decoders, however, never looked inside the model, and they
will serve unchanged when a better one supplies the distribution.

With this the first part of the book ends, and it ends on a limit. Our last
model predicts text one token at a time at a constant cost, and it makes
use of about three tokens of what it has read
(that section), because its gradients reach hardly further
(that section). There are two ways out. One keeps the fixed-size
state and redesigns the recurrence so that the gradient survives
(that section). The other gives up the fixed size: it keeps
every past token and lets the model look back at all of them, at a cost
that grows with the length of the text. This is attention
(that section), and today's language models are built on it.
Whichever is chosen, the tokenizer, the bits per byte, and the decoders of
this chapter carry over as they are.

## Summary

Decoding turns a language model's next-token distributions into sequences.
Maximization targets the most probable output: greedy decoding takes the
most probable token at each step and is myopic, and beam search keeps the
$w$ best prefixes at $w$ times the cost. Maximization suits tasks with few
acceptable outputs, but on open-ended text its output is repetitive.

Sampling draws from the model's distribution, reshaped by temperature
the equation and truncated by top-$k$ (a fixed count), top-$p$
(a fixed probability mass), or min-$p$ (a fixed fraction of the top
probability). On our model the truncated samplers write text as varied as
pure sampling and more probable. Perplexity evaluates the model, not its
samples, and generating naively re-runs the model over the whole prefix for
every new token, which a carried recurrent state avoids.

## Exercises

1. [code] **Repetition penalty.** The *repetition penalty* of
   @Keskar.McCann.Varshney.ea.2019 discourages tokens that already
   appear in the sequence by dividing each such token's logit by a constant
   $\gamma > 1$. Add it to `sample_next` so that it acts on the logits
   before both the greedy argmax and the temperature scaling. `generate`
   then has to pass the ids so far to `sample_next`. Decode
   `'The Time Traveller'` and `'It seemed to me that'` greedily with
   $\gamma = 1.2$.
    1. Does the penalty remove the repeating cycles? Report the distinct-3
       of each continuation, computed with `distinct`, with and without
       the penalty.
    1. Adding the same constant to every logit leaves the softmax
       unchanged. Does it leave the penalized distribution unchanged? Work
       out what the penalty does to a token whose logit is negative.
    1. Which legitimate repetitions in English text does the penalty
       discourage, and what happens as $\gamma$ grows large?

1. **Beam search properties.**
    1. Take beam width 2, the five-token vocabulary of
       the figure, and three steps. Assign conditional
       probabilities for which beam search does not return the most
       probable three-token sequence, and compute the probabilities of
       both sequences. What does your construction imply about how beam
       search can miss the most probable sequence?
    1. What is the smallest beam width that guarantees that beam search
       over $T$ steps returns the most probable of all
       $|\mathcal{V}|^T$ sequences, whatever the model?
    1. Let $w \leq |\mathcal{V}|$, and assume that no end-of-sequence
       token occurs. Count the calls to `step_fn` that `beam_search`
       makes over $T$ steps. Count also the calls that exhaustive search
       makes when it evaluates every prefix once. Evaluate both counts
       for $|\mathcal{V}| = 1{,}027$, $T = 50$, and $w = 4$.

1. [code] **Length normalization.** Run `beam_search` with $w = 4$ for up
   to 40 tokens after `'The Time Traveller'` and `'It seemed to me that'`.
   Pass the id of the newline character, `data.tokenizer.encode('\n')[0]`,
   as `eos_id`, and vary $\beta$ (the argument `alpha`) over
   $\{0, 0.75, 1.5\}$. Compare the lengths and the total log-probabilities
   of the winning candidates, and explain the trend using the equation.

1. **Truncation under temperature.** `sample_next` applies every
   cutoff to the temperature-scaled distribution $P_\tau$
   of the equation. Write the min-$p$ condition
   $P_\tau(x \mid x_{<t}) \geq p_{\min} \max_{x'} P_\tau(x' \mid x_{<t})$ as
   a condition on the untempered probabilities. For the five-token
   distribution 0.45, 0.25, 0.15, 0.10, 0.05, compute the sets that
   top-$k$ with $k = 2$, top-$p$ with $p = 0.85$, and min-$p$ with
   $p_{\min} = 0.5$ keep at $\tau = 2$. Which of the three rules keeps the
   same set at every temperature?

1. [code] **Constrained sampling.** Ban every token whose text contains the
   letter "e" by setting its logit to $-\infty$ in a wrapper around
   `step_fn`. Generate 50 tokens greedily after `'The Time Traveller'`
   with and without the ban. Report the probability mass that the ban
   removes, averaged over the steps of the banned continuation, and
   describe how the two continuations differ.
    1. Why is banning a *word* harder than banning a token under a subword
       tokenizer, and what can go wrong at the boundary between two
       tokens?
    1. Grammar and schema constraints are usually enforced by this same
       per-step masking, with a parser deciding at each step which tokens
       keep the output valid. Some constraints span several tokens, such
       as forcing a given phrase to appear somewhere in the output. What
       does keeping several prefixes alive, as beam search does, add over
       masking one token at a time for such a constraint?

1. [code] **Top-$p$ versus min-$p$.** ● At $\tau = 1.5$, sweep $p$ for
   top-$p$ and $p_{\min}$ for min-$p$ over a grid of values. For each
   setting, generate 50-token continuations of five prefixes of your
   choice and record two averages over the generated tokens: the number
   of tokens the rule keeps, and the log-probability of the drawn token
   under the untempered model. Plot the second average against the
   first, one curve per rule. Which rule draws more probable tokens at
   the same average number of kept tokens? Compare your finding with
   those of @Nguyen.Baker.Neo.ea.2025 .

1. [code] **Failure taxonomy.** ● Choose 20 prefixes and decode a 50-token
   greedy continuation of each.
    1. Sort the failures into a taxonomy of your own, with categories such
       as a repeating cycle or a drift away from the prefix's topic, and
       report the frequency of each category with one example.
    1. Decode one continuation of each prefix with min-$p$ sampling at
       $p_{\min} = 0.05$ and classify its failures with the same
       taxonomy, adding categories where needed. Which categories does
       sampling remove, and which does it introduce?
    1. Which of the automated statistics of
       that section would detect each
       category, and which categories need a human judge?

    *Adapted from Stanford CS224n,
    [Assignment 4](https://web.stanford.edu/class/archive/cs/cs224n/cs224n.1234/assignments/a4.pdf),
    Part 2(b).*

1. [code] **Constant cost per token.** ● `step_fn` re-runs the network over
   the entire prefix for every generated token, so a $T$-token
   continuation costs $O(T^2)$ cell
   updates (that section).
    1. Write a stateful replacement for `step_fn` that keeps the hidden
       state of `model.rnn` and consumes only the newest token, so that
       each call performs one cell update. Verify that `generate` produces
       the same greedy continuation with both versions.
    1. Time `generate` for 50, 200, and 800 tokens with both versions.
       Estimate from the ratios how each time grows with the number of
       tokens, and explain any departure from the quadratic and linear
       growth that the count of cell updates predicts.
    1. Why does the same replacement not carry over directly to
       `beam_search`, and what bookkeeping would make it work?

[Discussions](https://d2l.discourse.group/t/338)